In [ ]:
import logging
import os
import pickle
import random
import sys
from os.path import join

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# from hyperopt import fmin, tpe, hp, Trials, rand
import xgboost as xgb
from Bio import SeqIO
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

/home/hanxd/Repositories/ESP/Screening_and_application


In [ ]:
zeadata = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data/screening1", "dfzea.pkl")
)
zeadelete = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data/screening1", "zea_deletedata.pkl")
)

glydata = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data/screening1", "dfgly.pkl")
)
glydelete = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data/screening1", "glycine_deletedata.pkl")
)

eridata = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data/screening1", "dferi.pkl")
)
eridelete = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data/screening1", "erigeron_deletedata.pkl")
)

In [3]:
allzea = pd.concat([zeadata, zeadelete], ignore_index=True)
allgly = pd.concat([glydata, glydelete], ignore_index=True)
alleri = pd.concat([eridata, eridelete], ignore_index=True)

In [ ]:
def create_input_and_output_data(df):
    X = ()
    y = ()
    for ind in df.index:
        emb = df["ESM1b"][ind]
        ecfp = np.array(list(df["ECFP"][ind])).astype(int)

        X = X + (np.concatenate([ecfp, emb]),)
        y = y + (df["Binding"][ind],)

    return (X, y)


feature_names = ["ECFP_" + str(i) for i in range(1024)]
feature_names = feature_names + ["ESM1b_" + str(i) for i in range(1280)]

In [ ]:
bst = pd.read_pickle(
    os.path.join(
        CURRENT_DIR, "..", "data", "our_data/screening1", "deletedatamodel.dat"
    )
)

In [ ]:
data_X, data_y = create_input_and_output_data(df=alleri)
dwant = xgb.DMatrix(
    np.array(data_X), label=np.array(data_y), feature_names=feature_names
)
y_test_pred = bst.predict(dwant)
alleri["scores"] = y_test_pred
alleri.to_csv(our_data + "screening1/p450plant_deletedata_eri1-9.csv")

In [ ]:
data_X, data_y = create_input_and_output_data(df=allzea)
dwant = xgb.DMatrix(
    np.array(data_X), label=np.array(data_y), feature_names=feature_names
)
y_test_pred = bst.predict(dwant)
allzea["scores"] = y_test_pred
allzea.to_csv(our_data + "screening1/p450plant_deletedata_zea1-9.csv")

In [ ]:
data_X, data_y = create_input_and_output_data(df=allgly)
dwant = xgb.DMatrix(
    np.array(data_X), label=np.array(data_y), feature_names=feature_names
)
y_test_pred = bst.predict(dwant)
allgly["scores"] = y_test_pred
allgly.to_csv(our_data + "screening1/p450plant_deletedata_gly1-9.csv")